# CNN classification on dogs and cats images

In [2]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import random
from keras.preprocessing.image import load_img
warnings.filterwarnings('ignore')

c:\Users\nisha\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [3]:
# loading images
base_path = "E:/Datasets/Cats-Dogs-2/PetImages"
input_path = []
label = []

for class_name in os.listdir(base_path):
    class_path = os.path.join(base_path, class_name)
    
    for file_name in os.listdir(class_path):
        
        if class_name == 'Cat':
            label.append('cat')
        else:
            label.append('dog')
        
        input_path.append(os.path.join(class_path, file_name))

print(input_path[0], label[0])

E:/Datasets/Cats-Dogs-2/PetImages\Cat\0.jpg cat


In [4]:
len(label)

25000

In [5]:
len(input_path)

25000

In [6]:
# creating a data frame
df = pd.DataFrame()
df['images'] = input_path
df['label'] = label
df = df.sample(frac=1).reset_index(drop=True)
df.head(10)

,images,label
0,E:/Datasets/Cats-Dogs-2/PetImages\Cat\1821.jpg,cat
1,E:/Datasets/Cats-Dogs-2/PetImages\Dog\7147.jpg,dog
2,E:/Datasets/Cats-Dogs-2/PetImages\Dog\2129.jpg,dog
3,E:/Datasets/Cats-Dogs-2/PetImages\Cat\844.jpg,cat
4,E:/Datasets/Cats-Dogs-2/PetImages\Dog\4528.jpg,dog
5,E:/Datasets/Cats-Dogs-2/PetImages\Cat\5493.jpg,cat
6,E:/Datasets/Cats-Dogs-2/PetImages\Dog\3306.jpg,dog
7,E:/Datasets/Cats-Dogs-2/PetImages\Dog\5775.jpg,dog
8,E:/Datasets/Cats-Dogs-2/PetImages\Dog\4254.jpg,dog
9,E:/Datasets/Cats-Dogs-2/PetImages\Cat\12272.jpg,cat


In [7]:
from PIL import Image

bad_images = []

for image in df['images']:
    try:
        img = Image.open(image)  
        img.verify()             
    except:
        bad_images.append(image)

len(bad_images)

2

### EDA

In [75]:
plt.figure(figsize=(25, 25))
temp = df[df['label'] == 1]['images']
start = random.randint(0, len(temp))
files = temp[start:start+25]

for index, files in enumerate(files):
    plt.subplot(5, 5, index+1)
    img = load_img(files)
    img = np.array(img)
    plt.imshow(img)
    plt.title("Dogs")
    plt.axis('off')

<Figure size 2500x2500 with 0 Axes>

In [76]:
plt.figure(figsize=(25, 25))
temp = df[df['label'] == 0]['images']
start = random.randint(0, len(temp))
files = temp[start:start+25]

for index, files in enumerate(files):
    plt.subplot(5, 5, index+1)
    img = load_img(files)
    img = np.array(img)
    plt.imshow(img)
    plt.title("Dogs")
    plt.axis('off')

<Figure size 2500x2500 with 0 Axes>

In [77]:
df['label'].value_counts()

label
dog    12500
cat    12500
Name: count, dtype: int64

In [78]:
# input split
from sklearn.model_selection import train_test_split
train, test = train_test_split(df,test_size=0.2, random_state=42)

In [79]:
# Creating Data Generator for images
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_generator = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_generator = ImageDataGenerator(rescale=1./255)

In [85]:
train_iterator = train_generator.flow_from_dataframe(train, x_col='images', y_col='label',
                                                     target_size=(128, 128), batch_size=1024, class_mode='categorical')

val_iterator = val_generator.flow_from_dataframe(test, x_col='images', y_col='label',
                                                     target_size=(128, 128), batch_size=1024, class_mode='categorical')

Found 20000 validated image filenames belonging to 2 classes.
Found 5000 validated image filenames belonging to 2 classes.


## Model Creation

In [86]:
from keras import Sequential
from keras.layers import Conv2D, MaxPool2D, Flatten, Dense

model = Sequential([
    Conv2D(16, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    MaxPool2D(2, 2),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPool2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPool2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPool2D(2, 2),
    Flatten(),
    Dense(512, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [90]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_16 (Conv2D)              │ (None, 126, 126, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_16 (MaxPooling2D) │ (None, 63, 63, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 61, 61, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_17 (MaxPooling2D) │ (None, 30, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (None, 28, 28, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_18 (MaxPooling2D) │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_19 (MaxPooling2D) │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 512)            │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,457,761 (9.38 MB)

 Trainable params: 2,457,761 (9.38 MB)

 Non-trainable params: 0 (0.00 B)

In [91]:
history = model.fit(train_iterator, epochs= 100, validation_data=val_iterator)

UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x0000029CB69F8180>